## Import thư viện và Khởi tạo 

In [2]:
import os
import json 
from pathlib import Path 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field 

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)

judge_llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.0 # Bắt buộc phải là 0 để chấm điểm khách quan
)
print("Khởi tạo Giám khảo Llama-3-70B thành công!")

Khởi tạo Giám khảo Llama-3-70B thành công!


## Chuẩn bị dữ liệu giả lập (Mock Data)

In [3]:
# Giả sử đây là câu hỏi do đồng đội bạn sinh ra
sample_quiz = {
    "question": "AI Assistant trong hệ thống REIS sử dụng kỹ thuật gì?",
    "options": [
        "A. RAG (Retrieval-Augmented Generation)",
        "B. Nấu ăn", # Đáp án này quá ngớ ngẩn (Bad Distractor)
        "C. Nhảy múa", # Đáp án này quá ngớ ngẩn
        "D. Máy sưởi"
    ],
    "correct_answer": "A",
    "context": "AI Assistant trong REIS sử dụng kỹ thuật Retrieval-Augmented Generation (RAG) kết hợp với LLM Gemini để trả lời câu hỏi."
}

print("Dữ liệu đầu vào:")
print(json.dumps(sample_quiz, indent=2, ensure_ascii=False))


Dữ liệu đầu vào:
{
  "question": "AI Assistant trong hệ thống REIS sử dụng kỹ thuật gì?",
  "options": [
    "A. RAG (Retrieval-Augmented Generation)",
    "B. Nấu ăn",
    "C. Nhảy múa",
    "D. Máy sưởi"
  ],
  "correct_answer": "A",
  "context": "AI Assistant trong REIS sử dụng kỹ thuật Retrieval-Augmented Generation (RAG) kết hợp với LLM Gemini để trả lời câu hỏi."
}


## Xây dựng Bộ não Chấm thi 

In [4]:
evaluator_template = """
Bạn là một Giám khảo chuyên gia thẩm định chất lượng các câu hỏi trắc nghiệm E-Learning.
Nhiệm vụ của bạn là chấm điểm câu hỏi trắc nghiệm (MCQ) sau đây dựa trên ngữ cảnh được cung cấp.

[NGỮ CẢNH TỪ TÀI LIỆU (CONTEXT)]: 
{context}

[CÂU HỎI TRẮC NGHIỆM]:
Câu hỏi: {question}
Các lựa chọn: {options}
Đáp án đúng: {correct_answer}

[TIÊU CHÍ ĐÁNH GIÁ]:
1. Tính chính xác (accuracy): Đáp án đúng ({correct_answer}) có thực sự đúng và có nằm trong [NGỮ CẢNH] không?
2. Chất lượng đáp án nhiễu (distractor_quality): Các đáp án sai có hợp lý và mang tính đánh lừa không? (Nếu đáp án sai quá ngớ ngẩn, không liên quan đến ngữ cảnh, hãy đánh giá là Tệ).

HÃY TRẢ VỀ KẾT QUẢ ĐÁNH GIÁ DƯỚI ĐỊNH DẠNG JSON TUYỆT ĐỐI THEO CẤU TRÚC SAU (Không có thêm bất kỳ chữ nào khác):
{{
    "accuracy_score": <1 đến 10>,
    "distractor_score": <1 đến 10>,
    "feedback": "<Lời nhận xét chi tiết và gợi ý cách sửa câu hỏi cho hay hơn>",
    "is_accepted": <true/false - Chỉ true nếu cả 2 điểm đều >= 7>
}}
"""

prompt = PromptTemplate(
    template=evaluator_template,
    input_variables=["context", "question", "options", "correct_answer"]
)


## Chạy thử và phân tích kết quả

In [5]:
# Gắn data vào prompt
final_prompt = prompt.format(
    context=sample_quiz["context"],
    question=sample_quiz["question"],
    options=str(sample_quiz["options"]),
    correct_answer=sample_quiz["correct_answer"]
)

print("Đang yêu cầu Giám khảo Llama-3-70B đánh giá...")
response = judge_llm.invoke(final_prompt)

# In kết quả (Loại bỏ các ký hiệu markdown ```json nếu có)
raw_json = response.content.strip().replace("```json", "").replace("```", "")
result = json.loads(raw_json)

print("\n📊 BÁO CÁO ĐÁNH GIÁ TỪ AI:")
print(f"- Tính chính xác (Accuracy): {result['accuracy_score']}/10")
print(f"- Chất lượng nhiễu (Distractor): {result['distractor_score']}/10")
print(f"- Lời phê: {result['feedback']}")
print(f"- Quyết định: {'✅ DUYỆT' if result['is_accepted'] else '❌ TỪ CHỐI (Cần sửa)'}")


Đang yêu cầu Giám khảo Llama-3-70B đánh giá...

📊 BÁO CÁO ĐÁNH GIÁ TỪ AI:
- Tính chính xác (Accuracy): 10/10
- Chất lượng nhiễu (Distractor): 2/10
- Lời phê: Đáp án đúng (A) chính xác và có trong ngữ cảnh. Tuy nhiên, các đáp án sai (B, C, D) quá ngớ ngẩn và không liên quan đến ngữ cảnh, làm giảm chất lượng của câu hỏi. Để cải thiện, nên chọn các đáp án sai có liên quan đến chủ đề AI hoặc kỹ thuật, ví dụ như 'B. Machine Learning', 'C. Deep Learning', 'D. Natural Language Processing'.
- Quyết định: ❌ TỪ CHỐI (Cần sửa)


In [7]:
import pandas as pd


evaluator_template = """
Bạn là một Chuyên gia Khảo thí (Assessment Expert) cấp cao tại một trường Đại học. 
Nhiệm vụ của bạn là thẩm định khắt khe một câu hỏi trắc nghiệm (MCQ) do hệ thống AI khác sinh ra.

[NGỮ CẢNH TỪ TÀI LIỆU (CONTEXT)]: 
{context}

[CÂU HỎI TRẮC NGHIỆM ĐƯỢC SINH RA]:
Câu hỏi: {question}
Các lựa chọn: {options}
Đáp án đúng: {correct_answer}

[NHIỆM VỤ ĐÁNH GIÁ 4 CHIỀU]:
Bạn hãy phân tích sâu câu hỏi này theo 4 tiêu chí sau. Thang điểm cho mỗi tiêu chí là từ 1 đến 5 (5 là xuất sắc, 1 là tệ hại):

1. Độ bám sát (Context Relevance): Câu hỏi có hoàn toàn dựa vào [NGỮ CẢNH] không, hay sử dụng kiến thức ngoài lề?
2. Độ chính xác của đáp án (Answer Correctness): Đáp án đúng ({correct_answer}) có thực sự đúng và không thể chối cãi dựa trên ngữ cảnh không?
3. Chất lượng đáp án nhiễu (Distractor Plausibility): Các đáp án sai có đủ độ khó để "đánh lừa" học sinh không học bài không? Chúng có cùng trường từ vựng/chuyên ngành với đáp án đúng không?
4. Không có manh mối ngữ pháp (Grammar/Clue Independence): Câu hỏi có lỡ để lộ manh mối ngữ pháp (ví dụ: số ít/số nhiều, từ khóa lặp lại) giúp học sinh dễ dàng đoán mò ra đáp án đúng không?

[YÊU CẦU ĐẦU RA]:
Hãy suy luận từng bước (Chain-of-Thought) và xuất kết quả DUY NHẤT dưới dạng JSON theo cấu trúc sau:
{{
    "analysis": {{
        "relevance_analysis": "<Phân tích tiêu chí 1>",
        "correctness_analysis": "<Phân tích tiêu chí 2>",
        "distractor_analysis": "<Phân tích tiêu chí 3>",
        "clue_analysis": "<Phân tích tiêu chí 4>"
    }},
    "scores": {{
        "relevance": <1-5>,
        "correctness": <1-5>,
        "distractor": <1-5>,
        "clue_independence": <1-5>
    }},
    "total_score": <Tổng điểm 4 tiêu chí (tối đa 20)>,
    "verdict": "<'Khuyên dùng' (nếu tổng >= 16 và không có điểm nào dưới 3) / 'Cần sửa' (nếu ngược lại)>",
    "suggested_revision": "<Viết lại toàn bộ câu hỏi và 4 đáp án sao cho hoàn hảo nhất (Nếu câu hỏi đã hoàn hảo, hãy giữ nguyên)>"
}}
"""

prompt = PromptTemplate(
    template=evaluator_template,
    input_variables=["context", "question", "options", "correct_answer"]
)

# Giả lập 2 câu hỏi (1 câu cực hay, 1 câu cực tệ)
quiz_dataset = [
    {
        "question": "AI Assistant trong hệ thống REIS sử dụng kỹ thuật gì?",
        "options": ["A. RAG", "B. Nấu ăn", "C. Nhảy múa", "D. Chạy bộ"], # Câu tệ: Nhiễu quá ngớ ngẩn
        "correct_answer": "A",
        "context": "AI Assistant trong REIS sử dụng kỹ thuật RAG."
    },
    {
        "question": "Thành phần nào trong REIS chịu trách nhiệm dự báo xu hướng AQI?",
        "options": ["A. Forecasting Branch", "B. Anomaly Branch", "C. Data Ingestion", "D. Cache Layer"], # Câu hay: Nhiễu rất hợp lý
        "correct_answer": "A",
        "context": "Dual-AI Engine gồm một nhánh phát hiện bất thường và một nhánh Forecasting dự báo xu hướng AQI."
    }
]

# Chạy vòng lặp chấm điểm toàn bộ bài thi
aggregated_metrics = {
    "context_relevance": 0,
    "answer_correctness": 0,
    "distractor_plausibility": 0,
    "formatting_quality": 0
}

results_log = []

print("⚖️ Bắt đầu chấm điểm tự động toàn bộ Quiz...")
for i, item in enumerate(quiz_dataset):
    final_prompt = prompt.format(
        context=item["context"], question=item["question"],
        options=str(item["options"]), correct_answer=item["correct_answer"]
    )
    
    # Gọi AI chấm điểm
    raw_json = judge_llm.invoke(final_prompt).content.strip().replace("```json", "").replace("```", "")
    result = json.loads(raw_json)
    
    # Quy đổi điểm từ thang 5 sang thang 1.0
    rel_score = result["scores"]["relevance"] / 5.0
    cor_score = result["scores"]["correctness"] / 5.0
    dis_score = result["scores"]["distractor"] / 5.0
    fmt_score = result["scores"]["clue_independence"] / 5.0
    
    aggregated_metrics["context_relevance"] += rel_score
    aggregated_metrics["answer_correctness"] += cor_score
    aggregated_metrics["distractor_plausibility"] += dis_score
    aggregated_metrics["formatting_quality"] += fmt_score
    
    results_log.append({
        "question": item["question"],
        "relevance": rel_score, "correctness": cor_score, "distractor": dis_score
    })

# Tính trung bình (Average)
num_q = len(quiz_dataset)
for key in aggregated_metrics:
    aggregated_metrics[key] = round(aggregated_metrics[key] / num_q, 4)

print("\n📊 TỔNG HỢP METRICS HỆ THỐNG QUIZ (Giống Ragas):")
print(aggregated_metrics)

print("\n📋 ĐIỂM CHI TIẾT TỪNG CÂU:")
print(pd.DataFrame(results_log))


⚖️ Bắt đầu chấm điểm tự động toàn bộ Quiz...

📊 TỔNG HỢP METRICS HỆ THỐNG QUIZ (Giống Ragas):
{'context_relevance': 1.0, 'answer_correctness': 1.0, 'distractor_plausibility': 0.6, 'formatting_quality': 1.0}

📋 ĐIỂM CHI TIẾT TỪNG CÂU:
                                            question  relevance  correctness  \
0  AI Assistant trong hệ thống REIS sử dụng kỹ th...        1.0          1.0   
1  Thành phần nào trong REIS chịu trách nhiệm dự ...        1.0          1.0   

   distractor  
0         0.4  
1         0.8  
